# Day 32 — Practice ANSWERS
### Getting Data Into Shape

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — pen and paper | 8 |
| **B** | Find the mistake in a short program | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### How to work

**Part A is pen-and-paper first.** The numbers are chosen to be easy. Write your answer,
*then* run the cell to check.

Three questions to keep asking yourself:

- Does this model measure **distance**? If yes, it needs scaling.
- Did I **split before** anything learned from the data?
- Does this new feature actually **move the score**?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("ready")

ready


---
# Part A — Work it out

Eight questions. Most need no computer.

**The three formulas:**

| Scaler | Formula |
|---|---|
| standard | (x − mean) ÷ sd |
| min-max | (x − min) ÷ range |
| robust | (x − median) ÷ IQR |

### A1 · Standard scaling by hand

Five numbers. Work out the mean and the sd, then scale them.

*(Use the population sd — divide by n, not n−1. That is what scikit-learn does.)*

In [2]:
x = np.array([10., 20., 30., 40., 50.])

print("mean :", x.mean())
print("sd   :", x.std())
print("scaled:", StandardScaler().fit_transform(x.reshape(-1, 1)).ravel().round(3))

mean : 30.0
sd   : 14.142135623730951
scaled: [-1.414 -0.707  0.     0.707  1.414]


**mean = 30, sd = 14.142** (which is √200).

Scaled: `[-1.414  -0.707   0.      0.707   1.414]`

Each one is `(x − 30) ÷ 14.142`. The middle value lands on exactly 0, because it *is* the
mean.

Two things that are always true after standard scaling:

- the average of the column becomes **0**
- the spread becomes **1**

And notice there is **no upper limit** — a value of 200 here would come out at +12.0.

### A2 · Min-max by hand

Same five numbers.

In [3]:
print("min:", x.min(), "  max:", x.max(), "  range:", x.max() - x.min())
print("scaled:", MinMaxScaler().fit_transform(x.reshape(-1, 1)).ravel().round(3))

min: 10.0   max: 50.0   range: 40.0
scaled: [0.   0.25 0.5  0.75 1.  ]


`[0.    0.25  0.5   0.75  1.  ]`

The smallest becomes 0, the largest becomes 1, everything else spaces out in between.
`(20 − 10) ÷ 40 = 0.25`.

**Notice what the whole thing depends on: just two numbers — the min and the max.** Hold
on to that. It is the reason min-max is fragile.

### A3 · Now add one wild value

Same five numbers, plus a sixth: **500**. What happens to the first five under min-max?

In [4]:
x2 = np.array([10., 20., 30., 40., 50., 500.])
scaled = MinMaxScaler().fit_transform(x2.reshape(-1, 1)).ravel()

print("scaled:", scaled.round(4))
print()
print("the five original values are now spread over just",
      round(scaled[:5].max() - scaled[:5].min(), 4))

scaled: [0.     0.0204 0.0408 0.0612 0.0816 1.    ]

the five original values are now spread over just 0.0816


`[0.      0.0204  0.0408  0.0612  0.0816  1.    ]`

The five original numbers are squeezed into a span of **0.0816** — about 8% of the range.
Before the outlier arrived they filled the whole 0 to 1.

**One value did that.** And it did not even have to be wrong — it just had to be big.

This is why min-max is the fragile one: its entire output is decided by the two values
most likely to be a typo.

### A4 · Robust scaling, same wild value

Try the same six numbers with `RobustScaler`, which uses the **median** and the **IQR**
(the width of the middle half of the data) instead.

In [5]:
q1, med, q3 = np.percentile(x2, [25, 50, 75])
print("median:", med, "   IQR:", q3 - q1)

rb = RobustScaler().fit_transform(x2.reshape(-1, 1)).ravel()
print("scaled:", rb.round(3))
print()
print("the five original values are now spread over",
      round(rb[:5].max() - rb[:5].min(), 3))

median: 35.0    IQR: 25.0
scaled: [-1.  -0.6 -0.2  0.2  0.6 18.6]

the five original values are now spread over 1.6


median = **35**, IQR = 47.5 − 22.5 = **25**

`[-1.    -0.6   -0.2    0.2    0.6   18.6 ]`

The five original numbers keep a spread of **1.6** — they are still clearly different from
each other. And the outlier is reported honestly as sitting **18.6 IQRs out**.

| | span of the 5 normal values |
|---|---|
| min-max | 0.08 |
| robust | **1.60** |

The median and the IQR do not move when one value explodes. That is the entire argument
for `RobustScaler`.

### A5 · Which of these needs scaling?

For each model, say yes or no — and why.

| | Model |
|---|---|
| a | k-nearest neighbours |
| b | Random forest |
| c | k-means clustering |
| d | Decision tree |

| | Answer | Why |
|---|---|---|
| a | **Yes** | It works purely by measuring distance between rows |
| b | **No** | A forest is just many trees, and trees ask "is x above 8?" |
| c | **Yes** | "Clustering" means grouping by distance |
| d | **No** | Making the numbers bigger does not change their order |

**The one-line test:** *does this model measure distance between rows, or take steps
towards an answer?* If yes, scale. If it only ever compares values to thresholds, do not
bother.

### A6 · Which of these could a model work out on its own?

For each new column, say whether a linear model could have found it by itself.

| | The new column |
|---|---|
| a | `maths + science` |
| b | `bill ÷ items` |
| c | `2 × age` |
| d | `is_weekend` (from a date) |

| | Could it? | Why |
|---|---|---|
| a | **Yes** | Adding columns together is exactly what a linear model does |
| b | **No** | It can add and multiply by weights, but never divide one column by another |
| c | **Yes** | Multiplying by a constant just changes the weight it learns |
| d | **No** | The date is a single big number; "is it Saturday or Sunday" is not reachable from it |

**So (a) and (c) are a waste of a column** — the model already had them in effect.
**(b) and (d) are genuinely new**, and they are the kind worth building.

The rule: if your new column is just the old ones **added together or multiplied by a
number**, it adds nothing. Divisions, dates and yes/no flags are where the value is.

### A7 · What order should these go in?

Put these four steps in the correct order:

- scale the columns
- hide 30% of the rows
- build a `price_per_item` column
- train the model

1. **Build `price_per_item`** — it only looks at one row at a time, so it cannot leak.
2. **Hide 30% of the rows** — before anything learns from the data as a whole.
3. **Scale the columns** — learning the mean from the kept rows only.
4. **Train the model.**

### Why steps 1 and 2 can go either way, but 2 and 3 cannot

Building a ratio uses **only that row**: `bill ÷ items` for row 7 needs nothing from row 8.
So it is safe before or after the split.

Scaling is different. The mean and sd are computed from **all the rows you give it**. Do
that before splitting and the hidden rows have already influenced your model.

**When in doubt, split first.** It is never wrong.

### A8 · How many columns is too many?

You have **200 rows**. After one-hot encoding and some feature engineering you have **180
columns**.

What will happen, and what should you do?

In [6]:
print("rows    :", 200)
print("columns :", 180)
print("rows per column:", round(200 / 180, 2))

rows    : 200
columns : 180
rows per column: 1.11


**Barely one row per column.** The model has almost one adjustable knob for every row of
data.

It will fit the rows it studied **almost perfectly** — and fall apart on anything new. It
has stopped finding patterns and started memorising rows.

**What to do, in order:**

1. **Drop the obviously useless columns.** One-hot columns for categories that appear once
   or twice are the usual culprit — group those into "OTHER".
2. **Use `SelectKBest`** to keep only the columns most related to the answer.
3. **Get more rows** if you possibly can. This is the real fix.

A rough guide people use: aim for **at least 10 rows per column**. With 180 columns you
would want around 1,800 rows.

---
# Part B — Find the mistake

Four short programs. Each one runs cleanly and gives a misleading answer.

### B1 · The score is terrible and nobody knows why

In [7]:
rng = np.random.default_rng(601)
n = 400
area  = rng.normal(900, 250, n).clip(300, 2000)        # square feet
baths = rng.integers(1, 4, n).astype(float)            # 1 to 3
rent  = 4000 + 22*area + 3000*baths + rng.normal(0, 2500, n)

X = np.column_stack([area, baths])
X_train, X_hidden, y_train, y_hidden = train_test_split(X, rent, test_size=.3, random_state=0)

model = KNeighborsRegressor(7).fit(X_train, y_train)   # <-- look here
print("score:", round(model.score(X_hidden, y_hidden), 4))

score: 0.6679


**The problem:** `KNeighborsRegressor` works entirely by measuring distance — and `area`
runs to 2,000 while `baths` stops at 3.

When the model adds up how far apart two flats are, the bathroom count contributes
almost nothing. It is effectively predicting rent from the area alone, and ignoring
bathrooms completely.

**The fix:** scale both columns first, so they speak at the same volume.

In [8]:
scaler = StandardScaler().fit(X_train)             # learn from the training rows only

fixed = KNeighborsRegressor(7).fit(scaler.transform(X_train), y_train)

print("without scaling:", round(model.score(X_hidden, y_hidden), 4))
print("with scaling   :", round(fixed.score(scaler.transform(X_hidden), y_hidden), 4))

without scaling: 0.6679
with scaling   : 0.8243


```
without scaling: 0.6679
with scaling   : 0.8243
```

A jump of **0.156**, from nothing but putting the two columns on a fair footing.

Note how quiet this failure is. There is no error, no warning, and 0.67 does not look
obviously broken. You would only notice if you knew to check.

### B2 · Somebody scaled a random forest

In [9]:
X_train, X_hidden, y_train, y_hidden = train_test_split(X, rent, test_size=.3, random_state=0)
scaler = StandardScaler().fit(X_train)

plain  = RandomForestRegressor(n_estimators=80, random_state=0).fit(X_train, y_train)
scaled = RandomForestRegressor(n_estimators=80, random_state=0).fit(
             scaler.transform(X_train), y_train)

print("forest without scaling:", round(plain.score(X_hidden, y_hidden), 4))
print("forest with scaling   :", round(scaled.score(scaler.transform(X_hidden), y_hidden), 4))

forest without scaling: 0.7942


forest with scaling   : 0.7942


```
forest without scaling: 0.7942
forest with scaling   : 0.7942
```

**Identical.** This one is not a bug that breaks anything — it is **wasted effort and
wasted clarity**.

A tree asks questions like *"is the area above 900?"* Scaling changes 900 into 0.2, so the
tree asks *"is the area above 0.2?"* instead. **Same split, same rows on each side, same
tree.**

The real cost is to you, not the model: scaled numbers are much harder to read when you
look at what the tree actually learned.

**The fix:** if you are using a tree or a forest, skip the scaler.

### B3 · The scaler saw rows it should not have

In [10]:
rng = np.random.default_rng(602)
n = 400
Xb = rng.normal(50, 12, (n, 2))
yb = Xb[:, 0]*0.6 + Xb[:, 1]*0.3 + rng.normal(0, 4, n)

Xb_scaled = StandardScaler().fit_transform(Xb)        # <-- look here

a, b, c, d = train_test_split(Xb_scaled, yb, test_size=.3, random_state=0)
print("score:", round(KNeighborsRegressor(7).fit(a, c).score(b, d), 4))

score: 0.7437


**The problem:** the scaler was fitted on **every row**, including the 30% we were about
to hide. The mean and sd handed to the model were partly computed from the hidden rows, so
those rows were never truly hidden.

**The fix:** split first, then fit the scaler on the training rows only.

In [11]:
# the leaky version again, so we can compare the two side by side
a, b, c, d = train_test_split(Xb_scaled, yb, test_size=.3, random_state=0)
leaky = KNeighborsRegressor(7).fit(a, c).score(b, d)

# the honest version: split FIRST, then fit the scaler on the kept rows only
a, b, c, d = train_test_split(Xb, yb, test_size=.3, random_state=0)
sc = StandardScaler().fit(a)
honest = KNeighborsRegressor(7).fit(sc.transform(a), c).score(sc.transform(b), d)

print("scaled everything first   :", round(leaky, 4))
print("scaled the kept rows only :", round(honest, 4))
print("difference                :", round(leaky - honest, 4))

scaled everything first   : 0.7437
scaled the kept rows only : 0.743
difference                : 0.0007


```
scaled everything first   : 0.7437
scaled the kept rows only : 0.7430
difference                : 0.0007
```

### Now the honest part

The difference is **seven ten-thousandths**. A mean computed from 400 rows barely changes
when you remove 120 of them, so the leak had almost nothing to leak.

**So why make a fuss about it?**

1. It costs you **nothing** to do correctly — one extra line.
2. It is the **same keystroke** as the mistake in B4, which is catastrophic. If you let
   yourself fit scalers on everything, you will eventually fit a *column picker* on
   everything.
3. With `MinMaxScaler` and one extreme value in the hidden rows, it stops being small.

Report the honest number. Never defend a leak by saying it was tiny.

### B4 · The same mistake, four hundred times more expensive

In [12]:
rng = np.random.default_rng(603)
X_junk = rng.normal(0, 1, (120, 2500))
y_junk = rng.normal(0, 1, 120)            # nothing on earth predicts this

# pick the ten most promising columns, using ALL the rows
best = SelectKBest(f_regression, k=10).fit(X_junk, y_junk).get_support()

print("score:", round(cross_val_score(Ridge(), X_junk[:, best], y_junk, cv=5).mean(), 3))
print("(remember: this data is pure random noise. The honest answer is 0.)")

score: 0.433
(remember: this data is pure random noise. The honest answer is 0.)


**The problem:** the column picker searched all 2,500 columns **using every row**, and
only then did we test.

With 2,500 random columns and 120 rows, some will match the target by pure luck. The
picker found the ten luckiest — and it was allowed to look at the test rows while doing it.

By the time `cross_val_score` ran, the test rows had already helped choose the columns.

**The fix:** put the picker **inside** the pipeline, so it is re-run on each training split
separately.

In [13]:
pipe = Pipeline([
    ("pick",  SelectKBest(f_regression, k=10)),
    ("model", Ridge()),
])
honest = cross_val_score(pipe, X_junk, y_junk, cv=5).mean()

print("picker ran BEFORE the split :", round(cross_val_score(Ridge(), X_junk[:, best], y_junk, cv=5).mean(), 3))
print("picker ran INSIDE the split :", round(honest, 3))

picker ran BEFORE the split : 0.433
picker ran INSIDE the split : -0.186


```
picker ran BEFORE the split : +0.433
picker ran INSIDE the split : -0.186
```

Inside the pipeline the score collapses to **negative** — worse than guessing the average,
which is the truth about data with no signal in it.

### Compare B3 and B4

| | What leaked | How much it cost |
|---|---|---|
| **B3** | a scaler (2 numbers: mean and sd) | about **0.0007** |
| **B4** | a column picker searching 2,500 columns | about **0.62** |

**It is the same mistake.** The difference is how many chances the leaking step had to get
lucky. A scaler estimates two numbers. A picker buys 2,500 lottery tickets — and it pays
for them with your test rows.

---
# Part C — Write the code

Five tasks on one dataset.

In [14]:
# ---------------------------------------------------------------
# 800 food deliveries. Run this once - C1 to C5 all use `orders`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3232)
n = 800

items    = rng.integers(1, 15, n).astype(float)
per_item = rng.normal(190, 70, n).clip(50, None)
bill     = (items * per_item).round(0)
distance = rng.uniform(0.5, 12, n).round(2)                 # km
ordered  = pd.to_datetime("2025-03-01") + pd.to_timedelta(rng.integers(0, 24*90, n), "h")

is_dinner = ((ordered.hour >= 19) & (ordered.hour <= 22)).astype(float)
minutes   = (14 + 2.4*distance + 9*is_dinner + 0.02*items + rng.normal(0, 3.5, n)).round(1)

orders = pd.DataFrame({"bill": bill, "items": items, "distance_km": distance,
                       "ordered_at": ordered, "minutes": minutes})
orders.head()

,bill,items,distance_km,ordered_at,minutes
0,807.0,7.0,11.22,2025-03-26 07:00:00,44.3
1,905.0,4.0,11.02,2025-03-26 09:00:00,42.8
2,2835.0,13.0,10.03,2025-03-26 13:00:00,40.1
3,564.0,3.0,8.38,2025-05-27 05:00:00,34.7
4,1556.0,7.0,1.94,2025-03-13 09:00:00,16.9


### C1 · Does this model need scaling?

Fit a `KNeighborsRegressor(7)` to predict `minutes` from `bill`, `items` and
`distance_km` — **once without scaling and once with**. Report both scores on hidden rows.

Remember: split first.

In [15]:
# your code here

In [16]:
cols = ["bill", "items", "distance_km"]
X, y = orders[cols], orders["minutes"]

X_train, X_hidden, y_train, y_hidden = train_test_split(X, y, test_size=0.3, random_state=0)

# without scaling
plain = KNeighborsRegressor(7).fit(X_train, y_train)

# with scaling - learn the recipe on the training rows only
scaler = StandardScaler().fit(X_train)
scaled = KNeighborsRegressor(7).fit(scaler.transform(X_train), y_train)

print("without scaling:", round(plain.score(X_hidden, y_hidden), 4))
print("with scaling   :", round(scaled.score(scaler.transform(X_hidden), y_hidden), 4))

without scaling: -0.1251
with scaling   : 0.6763


```
without scaling: -0.1251
with scaling   :  0.6763
```

**From worse-than-guessing to a genuinely useful model**, and the only thing that changed
is the scaling.

A negative score means the model did worse than if it had ignored the inputs entirely and
answered "the average, 31 minutes" every single time. That is how badly an unscaled kNN
can fail.

Look at why. The `bill` column runs into the thousands; `distance_km` stops at 12. When
kNN measures how far apart two orders are, the bill dominates the sum completely — so the
model was finding "orders with a similar bill" and ignoring distance, which is the column
that actually drives the delivery time.

### C2 · Try all three scalers

Run the same kNN model with `StandardScaler`, `MinMaxScaler` and `RobustScaler`. Report
all three scores and say which you would choose here.

In [17]:
# your code here

In [18]:
for name, S in [("StandardScaler", StandardScaler()),
                ("MinMaxScaler  ", MinMaxScaler()),
                ("RobustScaler  ", RobustScaler())]:
    sc = S.fit(X_train)
    m = KNeighborsRegressor(7).fit(sc.transform(X_train), y_train)
    print(f"{name}: {m.score(sc.transform(X_hidden), y_hidden):.4f}")

StandardScaler: 0.6763
MinMaxScaler  : 0.6857
RobustScaler  : 0.6892


```
StandardScaler: 0.6763
MinMaxScaler  : 0.6857
RobustScaler  : 0.6892
```

**All three work, and they finish within 0.013 of each other.** That is the normal
situation, and it is the real answer to this question.

Put the two decisions side by side:

| Decision | What it was worth |
|---|---|
| scaling at all, versus not | **+0.80** (from −0.13 to 0.68) |
| which of the three scalers | **0.013** |

The first decision is about sixty times bigger than the second.

If you must choose, take **StandardScaler** — it is the sensible default, and this data
has no extreme outliers that would justify reaching for `RobustScaler`. But read nothing
into robust coming out on top by 0.013. Change `random_state=0` to `random_state=1` and
the order will probably shuffle. **That gap is noise, not evidence.**

Do not spend an afternoon choosing a scaler. Spend it on the thing in C3.

### C3 · Build two new features

Add these to the table, then measure whether each one helped:

1. `per_item` — the bill divided by the number of items
2. `is_dinner` — 1 if the order was placed between 7pm and 10pm, otherwise 0

Use scaled kNN, and add them **one at a time**.

In [19]:
# your code here

In [20]:
work = orders.copy()
work["per_item"]  = work["bill"] / work["items"]
work["is_dinner"] = ((work.ordered_at.dt.hour >= 19) &
                     (work.ordered_at.dt.hour <= 22)).astype(int)

def try_columns(column_list):
    """Scaled kNN on these columns. Returns the score on hidden rows."""
    a, b, c, d = train_test_split(work[column_list], work["minutes"],
                                  test_size=0.3, random_state=0)
    sc = StandardScaler().fit(a)
    m = KNeighborsRegressor(7).fit(sc.transform(a), c)
    return m.score(sc.transform(b), d)

base = ["bill", "items", "distance_km"]
previous = None
for extra in [[], ["per_item"], ["per_item", "is_dinner"]]:
    cols_now = base + extra
    s = try_columns(cols_now)
    label = "just the originals" if not extra else "+ " + extra[-1]
    change = "" if previous is None else f"   ({s - previous:+.4f})"
    print(f"{label:<20} {s:.4f}{change}")
    previous = s

just the originals   0.6763


+ per_item           0.6815   (+0.0052)


+ is_dinner          0.8201   (+0.1386)


```
just the originals   0.6763
+ per_item           0.6815   (+0.0052)
+ is_dinner          0.8201   (+0.1386)
```

**Two completely different results, and both are worth understanding.**

**`per_item` bought essentially nothing** — five thousandths, which is well inside the
wobble you would get from changing `random_state`. Why? Look back at how the data was
built: `minutes` depends on distance, dinner time and the item count — **not** on how
expensive the food is. There was never any signal in `per_item` to find.

Sit with that for a second, because `bill ÷ items` was the star example in the lecture.
**A ratio is not magic.** It helps when the thing it measures actually drives the answer,
and here it does not. Delete the column.

**`is_dinner` was transformative** — a jump of **0.139**, twenty-seven times bigger. And
it came out of `ordered_at`, a column that had been sitting unused in the table the whole
time.

**This is exactly why you add features one at a time.** Add both together and you watch
the score climb from 0.68 to 0.82 and conclude both were good. One of them is dead weight,
and you would carry it forever.

### C4 · Is a raw timestamp any use?

Add the timestamp as a plain number (seconds since the 1st of March) and check whether it
helps. Compare it against `is_dinner`.

In [21]:
# your code here

In [22]:
work["raw_time"] = (work.ordered_at - pd.Timestamp("2025-03-01")).dt.total_seconds()
work["hour"]     = work.ordered_at.dt.hour

base = ["bill", "items", "distance_km"]
for extra in ["raw_time", "hour", "is_dinner"]:
    print(f"{base} + {extra:<10}: {try_columns(base + [extra]):.4f}")

['bill', 'items', 'distance_km'] + raw_time  : 0.6864
['bill', 'items', 'distance_km'] + hour      : 0.7659
['bill', 'items', 'distance_km'] + is_dinner : 0.8297


```
+ raw_time  : 0.6864      (+0.010 over the 0.6763 baseline)
+ hour      : 0.7659      (+0.090)
+ is_dinner : 0.8297      (+0.153)
```

**The raw timestamp bought one hundredth of a point** — nothing, for a whole extra column.

A raw timestamp is just a counter that climbs forever: 7pm on Monday and 7pm on Tuesday
are twenty-four hours apart *as numbers*, even though they are the **same moment of the
day**. The repeating structure — the only part that matters here — is invisible to the
model.

Pull out the **hour** and the number comes back round: now 7pm on Monday and 7pm on
Tuesday are both just 19. Worth **+0.090**.

Ask the real question — **"is it dinner time?"** — and you get **+0.153**, because that is
exactly how the data was built.

**The same information, three times over.** All of it was sitting in `ordered_at` from the
start. What changed is how much work the model had to do to reach it.

**A date is never a feature.** What matters is the part of it that means something: the
hour, the weekday, whether it is a holiday, how long ago it was.

### C5 · Put the whole thing together

Build your best model: the useful features only, scaled properly, split properly. Report:

- the score on hidden rows
- the average error, in minutes
- how much of that error is irreducible *(look at how the data was built)*

In [23]:
# your code here

In [24]:
# per_item earned nothing in C3, so it does not go in.
best_cols = ["bill", "items", "distance_km", "is_dinner"]

X = work[best_cols]
y = work["minutes"]

X_train, X_hidden, y_train, y_hidden = train_test_split(X, y, test_size=0.3, random_state=0)

scaler = StandardScaler().fit(X_train)            # LEARN on the kept rows only
model = KNeighborsRegressor(7).fit(scaler.transform(X_train), y_train)

X_hidden_scaled = scaler.transform(X_hidden)      # APPLY the same recipe
guesses = model.predict(X_hidden_scaled)

print(f"score on hidden rows : {model.score(X_hidden_scaled, y_hidden):.4f}")
print(f"average error        : {mean_absolute_error(y_hidden, guesses):.2f} minutes")
print(f"delivery times range from {y.min():.0f} to {y.max():.0f} minutes")
print()
print("We built the data with rng.normal(0, 3.5) of pure noise per order,")
print("so no model could do much better than about 3 minutes of error.")

score on hidden rows : 0.8297
average error        : 3.08 minutes
delivery times range from 5 to 56 minutes

We built the data with rng.normal(0, 3.5) of pure noise per order,
so no model could do much better than about 3 minutes of error.


```
score on hidden rows : 0.8297
average error        : 3.08 minutes
delivery times range from 5 to 56 minutes
```

*(This is the same 0.8297 as the last line of C4 — which is the point. `per_item` was the
only thing we dropped, and dropping it cost nothing.)*

**That is about as good as this data allows.**

Every delivery time was generated with `rng.normal(0, 3.5)` of pure randomness added on
top. No model, however clever, can predict random noise. An average error of **3.08
minutes** against a noise level of **3.5** means we have extracted essentially all of the
real signal — there is nothing left to find.

**This is the most useful question you can ask about any score:** *how good could
anything possibly be on this data?* Without that comparison, "3 minutes of error" is a
number with no meaning. Here it happens to be excellent.

On real data you estimate the floor from measurement precision, from repeated
observations of identical conditions, or from how well a human expert does the same job.

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Build five features and keep only the good ones

Below is a raw table of 1,000 online shoppers. A model using the raw columns scores
poorly, because the things that actually drive spending are **not** any single column.

**Your tasks:**

1. Engineer **five** new features. Before measuring each one, write a sentence saying what
   real-world idea it captures.
2. Add them **one at a time** and record the score after each.
3. Keep only the ones that earned their place, and report your final score.

Hints, in increasing order of giving it away: think about **per-visit** amounts; think
about **how long ago** something happened; think about what `first_order` really encodes.

In [25]:
rng = np.random.default_rng(777)
n = 1000

first_order = pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 900, n), "D")
days_known  = (pd.Timestamp("2025-09-01") - first_order).days.values.astype(float)
n_visits    = rng.poisson(days_known / 95 + 2).astype(float)
total_spent = n_visits * rng.normal(540, 160, n).clip(100, None)
n_returns   = rng.binomial(n_visits.astype(int), 0.11).astype(float)
support_msgs = rng.poisson(n_visits * 0.18 + 0.5).astype(float)

# the truth: value depends on SPEND RATE and BASKET SIZE, hurt by the RETURN RATE
basket    = np.divide(total_spent, n_visits, out=np.full(n, 540.0), where=n_visits > 0)
rate      = total_spent / np.maximum(days_known, 30)
ret_rate  = np.divide(n_returns, n_visits, out=np.zeros(n), where=n_visits > 0)

value = (900 + 380*rate + 2.1*basket - 1400*ret_rate - 40*support_msgs
         + rng.normal(0, 180, n)).round(0)

shoppers = pd.DataFrame({"first_order": first_order, "n_visits": n_visits,
                         "total_spent": total_spent.round(0), "n_returns": n_returns,
                         "support_msgs": support_msgs, "value": value})
shoppers.head()

,first_order,n_visits,total_spent,n_returns,support_msgs,value
0,2025-03-26,4.0,2950.0,0.0,2.0,9366.0
1,2024-07-03,4.0,2642.0,1.0,0.0,4310.0
2,2023-11-14,10.0,4009.0,1.0,2.0,3942.0
3,2023-12-11,9.0,6711.0,0.0,2.0,6542.0
4,2023-03-29,18.0,7161.0,1.0,0.0,4890.0


In [26]:
# the starting point: the raw number columns, scaled
raw_cols = ["n_visits", "total_spent", "n_returns", "support_msgs"]

def score_shoppers(column_list, data):
    a, b, c, d = train_test_split(data[column_list], data["value"],
                                  test_size=0.3, random_state=0)
    sc = StandardScaler().fit(a)
    return Ridge().fit(sc.transform(a), c).score(sc.transform(b), d)

print("starting score:", round(score_shoppers(raw_cols, shoppers), 4))

starting score: 0.2095


In [27]:
# your five features here

In [28]:
d = shoppers.copy()

# 1. days_known - how long they have been a customer.
#    A DATE is not a number. What matters is the DISTANCE from that date to today.
d["days_known"] = (pd.Timestamp("2025-09-01") - d["first_order"]).dt.days

# 2. avg_basket - how much they spend per visit.
#    Separates "big spender" from "frequent visitor" - total_spent confuses the two.
d["avg_basket"] = d["total_spent"] / d["n_visits"].replace(0, 1)

# 3. spend_rate - how much they spend per day of being a customer.
#    Someone who spent Rs50,000 in 3 months is worth far more than over 3 years.
d["spend_rate"] = d["total_spent"] / d["days_known"].clip(lower=30)

# 4. return_rate - returns as a FRACTION of visits.
#    3 returns out of 4 visits is a problem. 3 out of 60 is normal.
d["return_rate"] = d["n_returns"] / d["n_visits"].replace(0, 1)

# 5. msgs_per_visit - support cost per unit of business. Same logic as return_rate.
d["msgs_per_visit"] = d["support_msgs"] / d["n_visits"].replace(0, 1)

cols, previous = list(raw_cols), None
for feature in [None, "days_known", "avg_basket", "spend_rate", "return_rate", "msgs_per_visit"]:
    if feature:
        cols.append(feature)
    s = score_shoppers(cols, d)
    label = "the raw columns" if feature is None else "+ " + feature
    change = "" if previous is None else f"   ({s - previous:+.4f})"
    print(f"{label:<20} {s:.4f}{change}")
    previous = s

the raw columns      0.2095
+ days_known         0.6096   (+0.4002)
+ avg_basket         0.6377   (+0.0280)
+ spend_rate         0.9940   (+0.3563)
+ return_rate        0.9947   (+0.0007)
+ msgs_per_visit     0.9947   (-0.0000)


### The result

```
the raw columns      0.2095
+ days_known         0.6096   (+0.4002)
+ avg_basket         0.6377   (+0.0280)
+ spend_rate         0.9940   (+0.3563)
+ return_rate        0.9947   (+0.0007)
+ msgs_per_visit     0.9947   (-0.0000)
```

**The score went from 0.21 to 0.99, and not one new measurement was taken.** Every one of
those columns was built out of the four we already had.

**Two features did almost all of the work:**

- `days_known` **+0.400** — the single biggest jump, and it came from a column the model
  could not read at all. `first_order` as a raw date is meaningless; *how long ago* is
  everything.
- `spend_rate` **+0.356** — this is literally one of the two terms `value` was built from.
  A linear model can add columns and multiply them by weights, but it can never **divide**
  one column by another. So `total_spent ÷ days_known` was permanently out of reach, no
  matter how long you trained.

**Two earned a little, or nothing:**

- `avg_basket` **+0.028** — this *is* the other term `value` was built from, so why so
  small? Because of the **order you add things in**. Once `days_known` is present, the
  model can already get a long way with `total_spent` and the length of the relationship.
  Add `avg_basket` on its own to the raw columns instead and it is worth far more. The
  lesson: these numbers measure *what each feature adds on top of what is already there*,
  not how good the feature is in isolation.
- `msgs_per_visit` **+0.000** — earns nothing. `support_msgs` was already in the baseline
  as a raw count, and that was enough. **Delete it.**

### D2 · A model that scores 0.97. Prove it is worth nothing.

A colleague hands you the model below. It predicts electricity bills with a score of
**0.97** on rows it has never seen. The scaling is correct, the split is correct, the code
has no bugs.

**Your tasks:**

1. Run it and confirm the score.
2. Work out why it is useless. It is **not** a scaling mistake and **not** a split mistake.
3. Prove it: rebuild from the honest columns and report the real score.
4. Write two or three sentences you could send your colleague.

*Hint: look at each column and ask when its value actually becomes known.*

In [29]:
rng = np.random.default_rng(2468)
n = 600

rooms     = rng.integers(1, 7, n).astype(float)
area_sqft = (rooms * rng.normal(230, 50, n)).clip(250, None).round(0)
residents = np.clip(np.round(rooms * 0.8 + rng.normal(0, 0.9, n)), 1, 9)
has_ac    = rng.binomial(1, 0.45, n)

units = (55 + 16*rooms + 0.045*area_sqft + 14*residents + 90*has_ac
         + rng.normal(0, 18, n)).clip(20, None)        # units of electricity used
bill  = (units * 7.4 + 120 + rng.normal(0, 25, n)).round(0)

# the meter reading, recorded by the meter man at the end of the month
units_consumed = units.round(1)

homes = pd.DataFrame({"rooms": rooms, "area_sqft": area_sqft, "residents": residents,
                      "has_ac": has_ac, "units_consumed": units_consumed, "bill": bill})
homes.head()

,rooms,area_sqft,residents,has_ac,units_consumed,bill
0,2.0,472.0,1.0,0,128.0,1075.0
1,3.0,558.0,3.0,0,186.4,1502.0
2,4.0,758.0,3.0,1,258.8,2058.0
3,4.0,979.0,4.0,1,324.3,2537.0
4,3.0,908.0,3.0,1,293.2,2295.0


In [30]:
cols = ["rooms", "area_sqft", "residents", "has_ac", "units_consumed"]

a, b, c, d = train_test_split(homes[cols], homes["bill"], test_size=0.3, random_state=0)
sc = StandardScaler().fit(a)
m = Ridge().fit(sc.transform(a), c)

print(f"score on hidden rows : {m.score(sc.transform(b), d):.4f}")
print(f"average error        : ₹{mean_absolute_error(d, m.predict(sc.transform(b))):,.2f}")

score on hidden rows : 0.9982
average error        : ₹20.48


In [31]:
# your investigation here

In [32]:
# Step 1 - which column is the model actually using? Look at the weights.
weights = pd.DataFrame({"column": cols, "weight": m.coef_.round(1)})
print(weights.sort_values("weight", key=abs, ascending=False).to_string(index=False))

        column  weight
units_consumed   576.4
         rooms    13.2
        has_ac    12.9
     residents     5.9
     area_sqft     4.3


In [33]:
# Step 2 - how closely does each column track the answer?
print("how strongly each column moves with the bill:")
print(homes[cols].corrwith(homes["bill"]).sort_values(key=abs, ascending=False).round(3).to_string())

how strongly each column moves with the bill:
units_consumed    0.999
rooms             0.806
area_sqft         0.767
residents         0.728
has_ac            0.555


In [34]:
# Step 3 - the question that settles it:
#   WHEN does each of these values become known?
#
#   rooms, area_sqft, residents, has_ac  -> known about the house at any time
#   units_consumed                       -> read off the meter at the END of the month,
#                                           at the same moment the bill is calculated
#
# The bill IS the units multiplied by the tariff. This is not a prediction, it is a receipt.

honest = ["rooms", "area_sqft", "residents", "has_ac"]
a, b, c, d = train_test_split(homes[honest], homes["bill"], test_size=0.3, random_state=0)
sc = StandardScaler().fit(a)
hm = Ridge().fit(sc.transform(a), c)

print(f"honest score         : {hm.score(sc.transform(b), d):.4f}")
print(f"honest average error : ₹{mean_absolute_error(d, hm.predict(sc.transform(b))):,.2f}")
print(f"\nfor comparison, bills vary by about ₹{homes.bill.std():,.0f}")

honest score         : 0.9530
honest average error : ₹108.62

for comparison, bills vary by about ₹607


In [35]:
# Step 4 - and show that bill really is just units x tariff
ratio = (homes.bill - 120) / homes.units_consumed
print(f"(bill - 120) / units_consumed:  {ratio.min():.2f} to {ratio.max():.2f}")
print("The tariff we built in was 7.4 per unit. There it is.")

(bill - 120) / units_consumed:  6.64 to 7.88
The tariff we built in was 7.4 per unit. There it is.


### Why it is useless

`units_consumed` is the **meter reading**. And the bill is simply the units multiplied by
the tariff plus a fixed charge — you can see it in the data: `(bill − 120) ÷ units` sits
right around **7.4** for every single home.

So the model learned `bill = 7.4 × units + 120`. **That is arithmetic, not prediction.**

Now ask the question that settles it: **when do you need this prediction?**

You want to estimate someone's bill **before** the month ends — to warn them, to plan, to
budget. The meter is not read until the month is over, and at that moment the bill is
already printed. The one input the model relies on arrives **after** the answer it is
supposed to predict.

### Why no amount of careful coding would have caught it

Be clear about this. The split was correct. The scaler was fitted on the training rows
only. The code has no bug in it anywhere.

The problem is in **which column was chosen**, and no technique can check that for you.
Only thinking about where each number comes from, and when.

### The honest numbers

```
with the meter reading : 0.9982   average error ₹20
without it             : 0.9530   average error ₹109
```

Two things to notice.

**The score barely moves** — 0.998 to 0.953. If you only ever look at the score, this
leak is almost invisible. **The error tells the real story:** it goes up more than
five-fold, from ₹20 to ₹109.

And the honest model is **still good**. Against bills that vary by about ₹607, an average
error of ₹109 is a genuinely useful tool — and unlike the first one, it would actually
work in February to predict March, using nothing but facts you already know about the
house.

### The warning sign

**A near-perfect score is a bug report, not a result.** Whenever you see one, check three
things:

1. Is one column a rescaled copy of the answer? *(this case)*
2. Did the same rows end up in both the training and hidden sets?
3. Did you accidentally leave the answer itself among the inputs?

### The message to send

> The 0.998 is coming entirely from `units_consumed` — the bill is literally that number
> times the ₹7.4 tariff plus the standing charge, so the model is reading the answer
> rather than predicting it. The meter is only read at the end of the month, which is
> exactly when we no longer need a prediction. Rebuilt on the house attributes we know up
> front, it scores 0.95 with an average error around ₹109. That one is less impressive
> but it would actually work.

---
## Before you hand it in

- For every model: does it **measure distance**? If yes, did you scale?
- Did you **split before** the scaler learned anything?
- Did you use `.fit()` on the training rows and `.transform()` on both?
- For every new feature: did you add it **one at a time** and check the score?
- Did you **delete** the features that did not help?
- For every column: would its value be **known at the moment you need the prediction**?

Three numbers worth carrying away:

- Scaling took a kNN model from **−0.13 to 0.68**. The same scaling changed a random forest
  by **exactly nothing**.
- A leaked **scaler** cost **0.0007**. A leaked **column picker** cost **0.62**. The same
  mistake, nearly a thousand times the damage.
- Engineered features took a model from **0.21 to 0.99** — with no new data collected.
- And two of those features were worth nothing at all. You only knew because you added
  them **one at a time**.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*